In [2]:
!pip install -q google-genai gradio

In [3]:
import time
import gradio as gr
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options=types.HttpOptions(timeout=45000),
)

# One attempt per model, then fall through to the next
MODELS = ["gemini-flash-lite-latest", "gemini-flash-latest"]

# Key points are written by a human, so the AI can't invent them
CONCEPTS = {
    "variable": [
        "stores a value in memory",
        "has a name used to refer to it",
        "the value can change",
    ],
    "loop": [
        "repeats a block of code",
        "runs until a condition stops it or items run out",
        "avoids writing the same code many times",
    ],
    "function": [
        "a named, reusable block of code",
        "can take inputs (parameters)",
        "can return an output",
    ],
    "list": [
        "stores several items in one variable",
        "items are kept in order",
        "each item is reached by its position (index)",
    ],
}

GRADER_PROMPT = """You are a patient programming mentor.
A student explained the concept "{concept}" in their own words.
Check the explanation against these key points:
{points}

For EACH key point, say one of: Covered, Partly covered, or Missing. Add a few words of evidence from the student's text.
Judge the meaning, not keywords. Do not reward vague answers.
If the explanation only restates the key points without any example or own wording, mark it Partly covered and ask for an example.
If the student's explanation contains a factual error, begin the "What you did well" line with "Not quite:" and kindly say what is incorrect.
If it is correct but incomplete, praise what is correct and do not use "Not quite:".
Only praise things that are actually correct. If nothing is correct, do not praise anything.

Reply in exactly this format:
**Key points:** one line per point with its status
**What you did well:** one sentence
**Improve this:** exactly one question that nudges the student toward the biggest missing point. Do NOT state the missing point yourself."""

REGRADE_NOTE = """
The student has revised their explanation. Their previous attempt is given too.
Add one extra line at the end:
**Progress:** one sentence on what improved or what is still missing."""

def ask(system_prompt, user_message):
    for model in MODELS:
        try:
            response = client.models.generate_content(
                model=model,
                contents=user_message,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    temperature=0.2,
                    max_output_tokens=1500,
                ),
            )
            if response.text:
                return response.text
        except Exception as e:
            print(f"[{model}] failed: {str(e)[:120]}")
            time.sleep(3)
    return "All models are busy or over quota right now. Wait a minute and try again."

def keep_one_question(text):
    # If "Improve this" has more than one question mark, keep only the first.
    marker = "**Improve this:**"
    if marker not in text:
        return text
    head, tail = text.split(marker, 1)
    if "?" not in tail:
        return text
    first_q, rest = tail.split("?", 1)
    extra = ""
    if "**Progress:**" in rest:
        extra = "\n\n**Progress:**" + rest.split("**Progress:**", 1)[1]
    return head + marker + first_q + "?" + extra

def grade(concept, explanation, previous=None):
    if not explanation or len(explanation.strip()) < 10:
        return "Please write a longer explanation in your own words (at least a sentence)."
    points = "\n".join(f"- {p}" for p in CONCEPTS[concept])
    system = GRADER_PROMPT.format(concept=concept, points=points)
    message = f"STUDENT EXPLANATION:\n{explanation}"
    if previous and previous.strip():
        system += REGRADE_NOTE
        message = f"PREVIOUS ATTEMPT:\n{previous}\n\nREVISED EXPLANATION:\n{explanation}"
    return keep_one_question(ask(system, message))

print("Ready. Now run Cell 3.")

Ready. Now run Cell 3.


In [4]:
def classify(result):
    """Turn the grader's text into a simple label."""
    if "Not quite:" in result:
        return "wrong"
    key_section = result.split("**What you did well")[0]
    weak = key_section.count("Missing") + key_section.count("Partly covered")
    return "good" if weak == 0 else "partial"

# (concept, student answer, label I EXPECT)
# "good" = all covered, "partial" = correct but incomplete, "wrong" = factual error
TESTS = [
    ("variable", "A variable is a box that holds something.", "partial"),
    ("variable", "A variable has a name and stores a value in memory. I can change the value later.", "good"),
    ("loop", "A loop runs your code exactly one time and then stops.", "wrong"),
    ("list", "A list holds many things together.", "partial"),
    ("function", "A function is a named block of code I can reuse, and it can take inputs and give back a result.", "good"),
    ("function", "A function is a variable that stores numbers.", "wrong"),
    ("list", "A list can only hold numbers.", "wrong"),
    ("loop", "A loop repeats code, like printing hello ten times.", "partial"),
]

matches = 0
for concept, answer, expected in TESTS:
    result = grade(concept, answer)
    got = classify(result)
    ok = (got == expected)
    matches += ok
    print(f"[{'MATCH' if ok else 'DIFFERENT'}] {concept}: expected={expected}, got={got}")
    print(f"   answer: {answer}")
    print(f"   one question? {result.split('**Improve this:**')[-1].count('?') == 1}")
    print("-" * 40)
    time.sleep(2)

print(f"\n{matches} of {len(TESTS)} matched what I expected.")

[MATCH] variable: expected=partial, got=partial
   answer: A variable is a box that holds something.
   one question? True
----------------------------------------
[MATCH] variable: expected=good, got=good
   answer: A variable has a name and stores a value in memory. I can change the value later.
   one question? True
----------------------------------------
[MATCH] loop: expected=wrong, got=wrong
   answer: A loop runs your code exactly one time and then stops.
   one question? True
----------------------------------------
[MATCH] list: expected=partial, got=partial
   answer: A list holds many things together.
   one question? True
----------------------------------------
[MATCH] function: expected=good, got=good
   answer: A function is a named block of code I can reuse, and it can take inputs and give back a result.
   one question? True
----------------------------------------
[MATCH] function: expected=wrong, got=wrong
   answer: A function is a variable that stores numbers.
   

In [5]:
REPEATS = [
    ("variable", "A variable is a box that holds something."),
    ("loop", "A loop runs your code exactly one time and then stops."),
    ("list", "A list holds many things together."),
]

for concept, answer in REPEATS:
    labels = []
    for _ in range(3):
        labels.append(classify(grade(concept, answer)))
        time.sleep(2)
    print(f"{concept}: {labels}  ->  {'consistent' if len(set(labels)) == 1 else 'VARIED'}")

variable: ['partial', 'partial', 'partial']  ->  consistent
loop: ['wrong', 'wrong', 'wrong']  ->  consistent
list: ['partial', 'partial', 'partial']  ->  consistent


In [ ]:
def show(text):
    return text.replace("\n", "\n\n")

def first_grade(concept, explanation):
    return show(grade(concept, explanation))

def regrade(concept, first_explanation, revised):
    if not revised or not revised.strip():
        return "Please write your revised explanation first."
    return show(grade(concept, revised, previous=first_explanation))

with gr.Blocks(title="Explain-It-Back Evaluator") as demo:
    gr.Markdown(
        "# Explain-It-Back Evaluator\n"
        "1. Pick a concept and explain it in your own words  \n"
        "2. Get feedback on what you covered and what is missing  \n"
        "3. Revise your answer and get graded again"
    )

    concept = gr.Dropdown(list(CONCEPTS.keys()), value="variable", label="Concept")
    explanation = gr.Textbox(label="Your explanation", lines=5)
    grade_btn = gr.Button("1. Grade my explanation", variant="primary")
    gr.Markdown("**Feedback**")
    feedback = gr.Markdown()

    revised = gr.Textbox(label="Your revised explanation", lines=5)
    regrade_btn = gr.Button("2. Re-grade my revision")
    gr.Markdown("**New feedback**")
    feedback2 = gr.Markdown()

    grade_btn.click(first_grade, [concept, explanation], feedback)
    regrade_btn.click(regrade, [concept, explanation, revised], feedback2)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4d4af07d4a9e00c6de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
